# Configuração do Ambiente:
Instalação de pacotes e depedências.  

In [0]:
#importação das libs
import basedosdados as bd
import pandas as pd
import numpy as np
from datetime import datetime, timezone
from google.cloud import storage
from google.cloud import bigquery
from google.oauth2 import service_account
import openpyxl
import pyarrow
from io import BytesIO
import os
from colorama import Fore, Style, init
from tabulate import tabulate


# Definição dos objetos: 

In [0]:
# Configurações do projeto e dataset
BILLING_ID = 'avaliacao-alfabetizacao-inep'
DATASET_ID = 'br_inep_avaliacao_alfabetizacao'

In [0]:
# tabelas do db avaliação alfabetização do inep
TABELAS = ["uf","meta_alfabetizacao_brasil", "meta_alfabetizacao_uf", "meta_alfabetizacao_municipio","municipio","alunos"]

#dicionário para armazenar as dataframes
dfs = {}

# Conjuntos de dados:

In [0]:
# 1. Caminho da sua chave no Databricks
CHAVE_PATH = "/Volumes/workspace/default/inep_avaliacao_alfabetizacao/avaliacao-alfabetizacao-inep-key.json"

# 2. Configura as credenciais
CREDENTIALS = service_account.Credentials.from_service_account_file(CHAVE_PATH)

# 3. Inicializa o cliente oficial do BigQuery
CLIENT = bigquery.Client(credentials=CREDENTIALS, project=BILLING_ID)

dfs = {}

# 4. Execução do loop para importar as tabelas
for i in TABELAS:
    print(f"Importando a tabela: {i}...")
    
    # Query que consulta os dados públicos da Base dos Dados no BigQuery
    query = f"SELECT * FROM `basedosdados.{DATASET_ID}.{i}`"
    
    # Baixa os resultados diretamente para um DataFrame do Pandas
    dfs[i] = CLIENT.query(query).to_dataframe()

print("Importação concluída com sucesso!")

# Verificação dos dados:

In [0]:
## definindo dicionário
dados = {
    "uf": dfs["uf"],
    "municipio": dfs["municipio"],
    "meta_alfabetizacao_uf": dfs["meta_alfabetizacao_uf"],
    "meta_alfabetizacao_municipio": dfs["meta_alfabetizacao_brasil"],
    "meta_alfabetizacao_brasil": dfs["meta_alfabetizacao_brasil"],
    "alunos": dfs["alunos"]
}

### Monitoramento:

In [0]:
# ============================================================
# INSPEÇÃO DOS DADOS BRUTOS
# ============================================================

print(f"{Fore.YELLOW} PRÉVIA DOS DADOS BRUTOS (Bronze)\n")

for entity, df in dados.items():
    print(f"{Fore.CYAN}{'─'*50}")
    print(f"{Fore.CYAN}  Entidade: {entity.upper()}")
    print(f"   Shape: {df.shape}")
    print("")
    print(f"   Tipos: { {col: str(dtype) for col, dtype in df.dtypes.items()} }")
    print("")
    print(f"   Nulos: {df.isnull().sum().to_dict()}")
    print()
    display(df.head(3))
    print()

# Metadados de Ingestão:

### Criação das variáveis de metadados:

In [0]:
#momento da ingestão
INGESTION_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
INGESTION_TIMESTAMP

#data da ingestão
INGESTION_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")
INGESTION_DATE

In [0]:
#origem dos dados
for i in TABELAS:
    dfs[i]['_momento_ingestao'] = INGESTION_TIMESTAMP
    dfs[i]['_data_ingestao'] = INGESTION_DATE
    dfs[i]['_origem'] = DATASET_ID + f".{i}"
    print(f"tabela {i} atualizada")

In [0]:
for i in TABELAS:
    print(f"---------Colunas da tabela {i}: ")
    print(dfs[i].columns)

# Ingestão dos dados na camada bronze (bruta):

In [0]:

# Folder em que as tabelas serão inseridas
dir_bronze = "/Volumes/workspace/default/inep_avaliacao_alfabetizacao/bronze/batch/"

# Garantindo que o diretório exista
os.makedirs(dir_bronze, exist_ok=True)


#ingestão das tabelas
for i in TABELAS:
    # 1. Usa o nome da tabela 'i' no nome do arquivo .parquet
    file_path = os.path.join(dir_bronze, f"{i}.parquet")

    # 2. Salva o DataFrame correspondente em formato Parquet
    dfs[i].to_parquet(file_path, engine="pyarrow", index=False)

    # 3. Imprime o caminho do arquivo salvo
    print(f"Tabela salva com sucesso em: {file_path}")
